# PoshanScan ? verified quarantine subset training (YOLOv8n-seg)

The latest audit isolated **40 of the original 120 images** for a small prototype run: **28 train / 7 validation / 5 test**, still assigned to their original partitions from seed **2026**. The other **80** remain quarantined because of marker gaps, contour/scope concerns, self-crossings, or the missing `train/110.png` label.

- One staged correction is applied: `val/54.png` drops an unrelated edge polygon; the kept mask is an exact raster match to the prior reviewed target.
- Three crop-only training masks (`train/14.png`, `26.png`, `44.png`) use your approved visible-region rule. No hidden anatomy was inferred or added.
- 36 other images were absent from the latest unresolved queue and were carried forward from prior visual review.
- **No SAM2 proposal is used as ground truth.** Original images, active labels, backups, and split files remain untouched.

Running all cells executes the strict preflight first, then starts a new pretrained `yolov8n-seg.pt` run for 100 epochs at image size 416. Training artifacts persist directly in Google Drive. The 5-image held-out test is only an exploratory synthetic-data check, not real-world or MUAC accuracy evidence. This workflow does not modify the API, MUAC calculations, or frontend.


In [ ]:
import sys, json, zipfile, hashlib, subprocess, shutil, csv
from pathlib import Path, PurePosixPath, PureWindowsPath
from datetime import datetime
from google.colab import drive
import torch, torchvision

drive.mount('/content/drive')
DRIVE_FOLDER = Path('/content/drive/MyDrive/PoshanScanColab')
TRAINING_RUNS_DIR = DRIVE_FOLDER / 'training_runs'
DATASET_ZIP = DRIVE_FOLDER / 'poshanscan_verified_quarantine_dataset.zip'
TRAINING_SCRIPT = DRIVE_FOLDER / 'train_poshanscan_colab.py'
EVALUATION_SCRIPT = DRIVE_FOLDER / 'evaluate_poshanscan_colab.py'
DATA_ROOT = Path('/content/poshanscan_yolo_dataset')
EXPECTED_SHA256 = {
    'dataset': '084919e0d1b9512b1e14999e64f9aa9b6c5565f9950a6b66542d7d7dfaa2bf67',
    'training_script': 'e1adfb1e98bf4761de15343f2e3efe24c59b7e5a9adc64a0c21c7429abdddd8b',
    'evaluation_script': '9da4aed88b2a436b9646c988f789c446c28c4ab8a30d4b3cfc964de453708d42',
}
TRAINING_RUNS_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

for key, path in {'dataset': DATASET_ZIP, 'training_script': TRAINING_SCRIPT, 'evaluation_script': EVALUATION_SCRIPT}.items():
    if not path.is_file():
        raise FileNotFoundError(f'Upload {path.name} to {DRIVE_FOLDER} first.')
    actual = sha256(path)
    if actual != EXPECTED_SHA256[key]:
        raise ValueError(f'{path.name} SHA-256 mismatch. Re-upload the exact project file; expected {EXPECTED_SHA256[key]}, got {actual}.')

print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU fallback selected; 100 epochs may take substantially longer.')
try:
    torchvision.ops.nms(torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]]), torch.tensor([.9,.8]), .5)
except Exception as exc:
    raise RuntimeError('This runtime PyTorch/torchvision pair failed NMS. Restart with a fresh Colab runtime; do not replace PyTorch here.') from exc
PREFLIGHT_PASSED = False
RUN_DIR = None


## Upload these files to Google Drive

Create `My Drive/PoshanScanColab/` and place these four files there:

1. `poshanscan_verified_quarantine_dataset.zip`
2. `train_poshanscan_colab.py`
3. `evaluate_poshanscan_colab.py`
4. `train_poshanscan_colab.ipynb`

The notebook verifies SHA-256 hashes for the dataset and both scripts before extraction. The ZIP contains only the selected images, labels, `dataset.yaml`, and split/selection manifests. It does not include SAM proposals or quarantined annotations.


## Extract the approved subset without changing its source split

This checks ZIP paths and integrity, then extracts to temporary Colab storage. The training script independently checks all image-label pairs and the complete original split manifest before any model is loaded.


In [ ]:
if DATA_ROOT.exists():
    raise FileExistsError(f'{DATA_ROOT} already exists. Restart the Colab runtime before re-extracting; this cell will not overwrite it.')
with zipfile.ZipFile(DATASET_ZIP) as archive:
    members = archive.namelist()
    if len(members) != 88 or len(members) != len(set(members)):
        raise ValueError(f'Expected 88 unique dataset files, got {len(members)}.')
    for name in members:
        posix, windows = PurePosixPath(name), PureWindowsPath(name)
        if posix.is_absolute() or windows.is_absolute() or '..' in posix.parts or '..' in windows.parts:
            raise ValueError(f'Unsafe ZIP member path: {name}')
    bad_member = archive.testzip()
    if bad_member:
        raise ValueError(f'ZIP CRC check failed for {bad_member}')
    archive.extractall(DATA_ROOT)

report = json.loads((DATA_ROOT / 'split_metadata' / 'split_report.json').read_text())
if report.get('dataset_profile') != 'verified_quarantine_subset' or report.get('seed') != 2026:
    raise ValueError('Unexpected dataset profile or source split seed.')
if report.get('source_split_counts') != {'train':84,'val':24,'test':12}:
    raise ValueError('Original split metadata is not 84/24/12.')
print('Profile:', report['dataset_profile'])
print('Included images by original partition:', report['split_counts'])
print('Quarantined:', report['quarantined_count'], '| source split regenerated:', report['source_split_regenerated'])


## Review the included and quarantined inventory

The following manifests are embedded in the ZIP. They let you inspect the precise selection and reasons without placing ambiguous labels in the training set.


In [ ]:
import pandas as pd
selection = pd.read_csv(DATA_ROOT / 'split_metadata' / 'dataset_selection_manifest.csv')
print('Decision totals:')
display(selection.groupby(['included','status']).size().rename('images').reset_index())
print('Included images:')
display(selection[selection['included'].astype(str).str.lower() == 'true'][['split','image_filename','status','reason']])
print('First quarantined items:')
display(selection[selection['included'].astype(str).str.lower() == 'false'][['split','image_filename','status','reason']].head(20))


## Strict preflight ? no training in this cell

This checks class ID 0, one polygon per included image, coordinate ranges, polygon area/self-crossings, image-label alignment, exact duplicates, all original split assignments, seed 2026, and duplicate-group isolation. It does not modify annotations or load a model.


In [ ]:
subprocess.run([
    sys.executable, str(TRAINING_SCRIPT),
    '--data-root', str(DATA_ROOT),
    '--seed', '2026',
    '--validate-only',
], check=True)
PREFLIGHT_PASSED = True
print('Preflight passed. Training may proceed on the verified 40-image subset.')


## Train and evaluate

This starts a fresh transfer-learning run from pretrained `yolov8n-seg.pt`; it does not load or resume the previous poor/lost checkpoint. The training script writes into Drive during training, uses a positive train batch and evaluation batch 1, evaluates the held-out test split, saves metrics/confusion matrices/prediction overlays/logs, and records checkpoint task/class compatibility for FastAPI.

Run this cell only after the preflight cell succeeds.


In [ ]:
START_TRAINING = True  # User requested a new run after passing preflight.
if not PREFLIGHT_PASSED:
    raise RuntimeError('Training blocked: run and pass the strict preflight first.')
if START_TRAINING:
    subprocess.run([
        sys.executable, str(TRAINING_SCRIPT),
        '--data-root', str(DATA_ROOT),
        '--training-runs-dir', str(TRAINING_RUNS_DIR),
        '--weights', 'yolov8n-seg.pt',
        '--epochs', '100', '--imgsz', '416', '--seed', '2026',
    ], check=True)
    run_dirs = sorted(TRAINING_RUNS_DIR.glob('yolov8n-seg-upper-arm-*'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not run_dirs:
        raise RuntimeError('Training command ended but no Drive run directory exists.')
    RUN_DIR = run_dirs[0]
    for required in ('weights/best.pt','weights/last.pt','results.csv','training.log','training_report.json','checkpoint_compatibility.json'):
        if not (RUN_DIR / required).is_file():
            raise FileNotFoundError(f'Training run is missing required artifact: {required}')
    print('Persisted run:', RUN_DIR)
else:
    print('Training switch is off.')


## Inspect persisted training and held-out test results

No metrics are fabricated. This cell displays files only if the training/evaluation scripts actually wrote them. The test partition has 5 images, so its estimates will be highly uncertain. All input examples are synthetic; these results cannot establish real-world segmentation or MUAC performance.


In [ ]:
from PIL import Image
from IPython.display import display
import pandas as pd
if RUN_DIR is None:
    candidates = sorted(TRAINING_RUNS_DIR.glob('yolov8n-seg-upper-arm-*'), key=lambda p: p.stat().st_mtime, reverse=True)
    RUN_DIR = candidates[0] if candidates else None
if RUN_DIR is None:
    print('No persisted training run exists yet.')
else:
    print('Run:', RUN_DIR)
    for name in ('training.log','results.csv','training_curves_live.png','training_report.json','checkpoint_compatibility.json'):
        path = RUN_DIR / name
        print(name, 'exists=', path.is_file(), 'bytes=', path.stat().st_size if path.is_file() else None)
    if (RUN_DIR / 'training_curves_live.png').is_file():
        display(Image.open(RUN_DIR / 'training_curves_live.png'))
    if (RUN_DIR / 'results.csv').is_file():
        display(pd.read_csv(RUN_DIR / 'results.csv').tail(10))
    for report_path in RUN_DIR.glob('evaluation/**/evaluation_report.json'):
        print(report_path.read_text())
    for image_path in list(RUN_DIR.glob('evaluation/**/*.png'))[:8] + list(RUN_DIR.glob('evaluation/**/*comparison.jpg'))[:8]:
        print(image_path.name)
        display(Image.open(image_path))


## Recover an interrupted run

If Colab disconnects after at least one completed epoch, remount Drive, re-extract this same ZIP in a fresh runtime, and pass preflight. Resume only from that interrupted run's `weights/last.pt`; the script checks image/label hashes and refuses a changed dataset. Do not resume from `best.pt`, and do not reuse the previous lost checkpoint.


In [ ]:
RESUME_AFTER_INTERRUPT = False
RESUME_CHECKPOINT = None  # Set to the interrupted Drive run's weights/last.pt
if not RESUME_AFTER_INTERRUPT:
    print('Resume disabled.')
elif not PREFLIGHT_PASSED:
    raise RuntimeError('Resume blocked until the same verified dataset passes preflight.')
else:
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)
    if RESUME_CHECKPOINT.name != 'last.pt' or not RESUME_CHECKPOINT.is_file():
        raise FileNotFoundError('Set RESUME_CHECKPOINT to an existing Drive weights/last.pt.')
    subprocess.run([
        sys.executable, str(TRAINING_SCRIPT),
        '--data-root', str(DATA_ROOT),
        '--training-runs-dir', str(TRAINING_RUNS_DIR),
        '--seed', '2026', '--resume', str(RESUME_CHECKPOINT),
    ], check=True)


## Evaluation-only recovery

If training produced `best.pt` and `results.csv` but evaluation failed, run only this cell after selecting that Drive run. It loads the existing checkpoint and uses fixed positive batch size 1; it never calls training.


In [ ]:
EVALUATE_ONLY_RECOVERY = False
COMPLETED_RUN_DIR = None  # Example: TRAINING_RUNS_DIR / 'yolov8n-seg-upper-arm-...'
if not EVALUATE_ONLY_RECOVERY:
    print('Evaluation-only recovery is disabled.')
else:
    COMPLETED_RUN_DIR = Path(COMPLETED_RUN_DIR)
    best = COMPLETED_RUN_DIR / 'weights' / 'best.pt'
    if not best.is_file() or not (COMPLETED_RUN_DIR / 'results.csv').is_file():
        raise FileNotFoundError('Expected best.pt and results.csv in the selected Drive run.')
    out = COMPLETED_RUN_DIR / ('evaluation_recovery_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    subprocess.run([
        sys.executable, str(EVALUATION_SCRIPT), '--weights', str(best),
        '--data-root', str(DATA_ROOT), '--training-run', str(COMPLETED_RUN_DIR),
        '--output-dir', str(out), '--imgsz', '416', '--batch', '1', '--api-imgsz', '640',
    ], check=True)
    print('Evaluation-only artifacts:', out)


## Download `best.pt`

Run this final cell after the Drive checkpoint exists.


In [ ]:
from google.colab import files
if RUN_DIR is None:
    candidates = sorted(TRAINING_RUNS_DIR.glob('yolov8n-seg-upper-arm-*'), key=lambda p: p.stat().st_mtime, reverse=True)
    RUN_DIR = candidates[0] if candidates else None
if RUN_DIR is None:
    raise FileNotFoundError('No Drive training run exists; training has not completed.')
best = RUN_DIR / 'weights' / 'best.pt'
if not best.is_file():
    raise FileNotFoundError(f'best.pt not found: {best}')
files.download(str(best))
